In [1]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier

In [2]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [3]:
data = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [4]:
data.head(3)

,participant,timestamp,clinical-timestamp,sis-01,sis-02,sis-03,sis-04,sis-05,sis-06,sis,...,sleep-duration-to-wakeup,sleep-wakeup-count,sleep-heartrate-mean,sleep-heartrate-min,sleep-heartrate-max,step-count,step-ratio,step-mean,step-max,step-max-timestamp
0,1,2022-03-31,2022-04-13,5,5,4,4,5,2,25,...,0.00,3,59,54,74,415,0.5000,34.5833,106,13
1,1,2022-04-01,2022-04-13,5,5,4,4,5,2,25,...,0.00,4,61,56,82,447,0.2917,63.8571,185,8
2,1,2022-04-02,2022-04-13,5,5,4,4,5,2,25,...,0.17,3,59,55,83,533,0.4231,56.1132,170,14


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 84 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   participant                            1008 non-null   int64  
 1   timestamp                              1008 non-null   object 
 2   clinical-timestamp                     1008 non-null   object 
 3   sis-01                                 1008 non-null   int64  
 4   sis-02                                 1008 non-null   int64  
 5   sis-03                                 1008 non-null   int64  
 6   sis-04                                 1008 non-null   int64  
 7   sis-05                                 1008 non-null   int64  
 8   sis-06                                 1008 non-null   int64  
 9   sis                                    1008 non-null   int64  
 10  ohs-01                                 1008 non-null   int64  
 11  ohs-

In [6]:
# Convert tensors to pandas DataFrame
df = data[["sis", "ohs", "oks"]]

In [7]:
df.head(3)

,sis,ohs,oks
0,25,25,31
1,25,25,31
2,25,25,31


In [8]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(df["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(df["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(df["oks"], [25, 50, 75])

In [9]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [10]:
# Apply discretization
df["SISS_Category_Q"] = pd.cut(df["sis"], bins=[df["sis"].min(), siss_q1, siss_q2, siss_q3, df["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
df["OHSS_Category_Q"] = pd.cut(df["ohs"], bins=[df["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, df["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

df["OKSS_Category_Q"] = pd.cut(df["oks"], bins=[df["oks"].min(), okss_q1, okss_q2, okss_q3, df["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [11]:
# Extract features, target, and patient IDs for LOPO
X = data.iloc[:, -46:] ## pd.DataFrame(samples)
y = df["OHSS_Category_Q"]
groups = data["participant"] ## pd.Series(participants)

In [12]:
#84-39 + 1

In [13]:
#X.info()

In [14]:
# Define classifier and hyperparameter grid
#model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
param_grid = {"max_depth": [3, 5, 7], 
              "n_estimators": [20, 50]}

In [15]:
len(y)

1008

In [16]:
len(X)

1008

In [17]:
len(groups)

1008

In [18]:
# Leave-One-Patient-Out CV (LOPO)
outer_logo = LeaveOneGroupOut()

In [19]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
performance_metrics = []

In [20]:
# Outer LOPO Loop
count=0

for train_idx, test_idx in outer_logo.split(X, y, groups):
    #print(train_idx) index
    count=count+1
    print(count)
    
    #if count == 6 or count == 7:
    #    continue 
    X_train_outer, X_test = X.iloc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups.iloc[train_idx]
    print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")
            # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters

    # Train best model on full outer training set
    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    # Compute metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average="macro")
    precision = precision_score(y_test, y_pred, average="macro")
    conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])

    # Aggregate confusion matrices
    overall_conf_matrix += conf_matrix

    # Store results
    performance_metrics.append([f1, balanced_acc, recall, precision])

1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
2
[ 1  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 

[ 1  3  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 16 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 16 17]
11
[ 1  2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 1  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 1  2  4  5  6  7  8  9 10 12 13 14 15 16 1

In [21]:
overall_conf_matrix

array([[ 11., 154.,   5.,  96.],
       [ 74., 169.,   9., 140.],
       [ 19.,  36.,   1.,  42.],
       [119.,  61.,  16.,  56.]])

In [22]:
# Convert results to DataFrame
performance_df = pd.DataFrame(performance_metrics, columns=["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])

# Save performance metrics and overall confusion matrix
output_path = os.path.join(root, "results/model_performance_ohss.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [23]:
performance_df

,Macro-F1,Balanced Accuracy,Macro Recall,Macro Precision
0,0.075758,0.178571,0.044643,0.250000
1,0.000000,0.000000,0.000000,0.000000
2,0.168889,0.339286,0.113095,0.333333
3,0.025424,0.053571,0.013393,0.250000
4,0.011696,0.017857,0.005952,0.333333
5,0.176056,0.446429,0.223214,0.145349
6,0.148656,0.261905,0.130952,0.172457
7,0.289260,0.392857,0.261905,0.351901
8,0.017241,0.035714,0.008929,0.250000
9,0.153846,0.428571,0.214286,0.120000


In [24]:
# Plot overall confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Overall Confusion Matrix OHSS")
conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance metrics saved as: {output_path}")
print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")


✅ Performance metrics saved as: .\results/model_performance_ohss.xlsx
✅ Overall Confusion Matrix saved as: .\results/overall_confusion_matrix_ohss.pdf
